# Jim-Tec / Arla butter cannon — exploratory data analysis

Interactive examination of the CSV logs in `data/jim-tec/arla/butter cannon/`.
Not part of any pipeline and decides nothing (D68): it is for looking. Choices made only for looking
(block lengths, Welch settings, filter bands, which files are shown) are set in the cell where they are used and
marked *EDA choice*. Unknowns that the notebook must not guess are marked **PLACEHOLDER**.

**What the files look like (checked by hand before writing this notebook)**
- Two columns: `Volt_CH0` (one voltage channel, sensor type not documented; `README.txt` is empty) and `RPM`.
- Logs from 2026-10-12 and 2026-10-13 start with `#` header lines (`SamplingRate_Hz=50000.00`,
  `LogDuration_minutes=15`, ...) and are ~45 M rows (~495 MB) each.
- Older logs (2026-04-14 to 2026-10-09) have **no header**, so their sampling rate is unknown here.
- `RPM` is stored per sample but changes only in steps of a few thousand samples (a slower update rate).

**Run:** `uv sync --group eda`, then open this notebook (VS Code or `uv run jupyter lab`).
Files are read one at a time (one 15-min file is ~360 MB in memory as float32).
Sections 4 and 6 each read every file once and take several minutes.

Sections
1. Setup  2. File table  3. One file up close  4. Block statistics over all files
5. RPM  6. Spectra and envelope spectra per operating point  7. Open questions

## 1. Setup

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, hilbert, sosfiltfilt, welch
from scipy.stats import kurtosis


def repo_root():
    """The repository root: the first folder upwards that contains pyproject.toml."""
    p = Path.cwd().resolve()
    while not (p / "pyproject.toml").exists():
        p = p.parent
    return p


ROOT = repo_root()
DATA = ROOT / "data" / "jim-tec" / "arla" / "butter cannon"
FILES = sorted(DATA.glob("log_*.csv"))

# PLACEHOLDER: sampling rate (Hz) of the logs without a header line. Unknown; not assumed.
# Leave None to treat those files in samples only (no time axis, no spectra).
FS_NO_HEADER = None

print(len(FILES), "CSV files in", DATA)

In [ ]:
def read_header(path):
    """The '# key=value' lines at the top of a log, as a dict of strings (empty if there are none)."""
    header = {}
    with open(path, "r") as fh:
        for line in fh:
            if not line.startswith("#"):
                break
            key, _, value = line[1:].strip().partition("=")
            header[key.strip()] = value.strip()
    return header


def sampling_rate(path):
    """Sampling rate in Hz from the header, else FS_NO_HEADER (may be None)."""
    h = read_header(path)
    return float(h["SamplingRate_Hz"]) if "SamplingRate_Hz" in h else FS_NO_HEADER


def start_time(path):
    """Start time from the file name log_YYYY-MM-DD_HH-MM-SS.csv."""
    return pd.to_datetime(re.search(r"log_(.+)\.csv", path.name).group(1), format="%Y-%m-%d_%H-%M-%S")


def read_log(path):
    """Voltage (float32) and RPM (float32; NaN where a row is incomplete) of one log."""
    df = pd.read_csv(path, comment="#", dtype={"Volt_CH0": np.float32, "RPM": np.float32}, engine="c")
    return df["Volt_CH0"].to_numpy(), df["RPM"].to_numpy()

## 2. File table
Header contents, start time (from the file name), size and sampling rate. Row counts need a full read
and are added in section 4. For the 15-min logs, the gap column shows whether one log starts where
the previous one ends (expected gap 0 s if logging is continuous).

In [ ]:
rows = []
for p in FILES:
    h = read_header(p)
    rows.append({"file": p.name, "start": start_time(p), "size_mb": p.stat().st_size / 1e6,
                 "fs_hz": sampling_rate(p), "duration_min_header": float(h["LogDuration_minutes"]) if "LogDuration_minutes" in h else np.nan,
                 "header_keys": ", ".join(h)})
files = pd.DataFrame(rows).set_index("file")
files["gap_to_prev_s"] = (files.start - (files.start.shift() + pd.to_timedelta(files.duration_min_header.shift(), unit="min"))).dt.total_seconds()
files

In [ ]:
# Full header of one log with a header.
for k, v in read_header(next(p for p in FILES if read_header(p))).items():
    print(f"{k}: {v}")

## 3. One file up close
Raw waveform and value distribution of one log. Change `FILE` to look at another one.

In [ ]:
FILE = DATA / "log_2026-10-12_08-57-58.csv"  # EDA choice: first 15-min log
v, rpm = read_log(FILE)
fs = sampling_rate(FILE)
print(f"{FILE.name}: {len(v):,} rows, fs = {fs} Hz" + (f", {len(v) / fs / 60:.2f} min" if fs else ""))
print("incomplete rows (RPM missing):", int(np.isnan(rpm).sum()), " voltage NaN:", int(np.isnan(v).sum()))
print(f"voltage min {v.min():.4f}  max {v.max():.4f}  mean {v.mean():.4f}  std {v.std():.4f}")
print("RPM values (count):", pd.Series(rpm).value_counts().head(10).to_dict())

**Whole file.** A 15-min file has 45 M samples, far more than the ~1,000 pixel columns of a plot, so each
pixel column covers ~45,000 samples. Drawing the min and max of every block of `ENV_N` samples gives the
same picture as plotting every sample (what a full plot shows per pixel column is exactly its min-to-max range),
but draws in a second instead of minutes. The black line is the block mean.

In [ ]:
ENV_N = 5_000  # EDA choice: samples per min/max block (0.1 s at 50 kHz)
n = len(v) // ENV_N * ENV_N
blocks = v[:n].reshape(-1, ENV_N)
x = np.arange(blocks.shape[0]) * ENV_N / (fs or 1) / (60 if fs else 1)
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].fill_between(x, blocks.min(1), blocks.max(1), lw=0, alpha=0.6, label="min/max per block")
ax[0].plot(x, blocks.mean(1), lw=0.6, color="k", label="mean per block")
ax[0].set_ylabel("Volt_CH0 [V]"); ax[0].legend(loc="upper right")
ax[1].plot(x, rpm[:n:ENV_N], lw=0.8)
ax[1].set_ylabel("RPM"); ax[1].set_xlabel("time [min]" if fs else "sample")
fig.suptitle(FILE.name); plt.tight_layout(); plt.show()

**Zooms.** The raw samples (every sample plotted) in windows of decreasing length, all starting in the
middle of the file, so the waveform shape (impulses, periodicity) is visible.

In [ ]:
ZOOMS_S = [1.0, 0.1, 0.01]  # EDA choice: window lengths in s
i0 = len(v) // 2
fig, ax = plt.subplots(len(ZOOMS_S), 1, figsize=(12, 2.5 * len(ZOOMS_S)))
for a, z in zip(ax, ZOOMS_S):
    seg = v[i0:i0 + int(z * (fs or 50_000))]
    a.plot(np.arange(len(seg)) / (fs or 1) * 1e3, seg, lw=0.5)
    a.set_title(f"{z * 1e3:g} ms from sample {i0:,} (RPM {rpm[i0]:.0f})"); a.set_ylabel("V")
ax[-1].set_xlabel("time [ms]" if fs else "sample")
plt.tight_layout(); plt.show()

**Value distribution and resolution.** The CSV stores 4 decimals; the printout gives the smallest step between
distinct values (the effective resolution of the logger) and the histogram shows how the values are spread.

In [ ]:
u = np.unique(v)
steps = np.diff(u)
print(f"distinct values: {len(u):,}; smallest step {steps.min():.6f} V; median step {np.median(steps):.6f} V")
plt.figure(figsize=(8, 3.5))
plt.hist(v, bins=400, log=True)
plt.xlabel("Volt_CH0 [V]"); plt.ylabel("count"); plt.title(FILE.name); plt.show()

## 4. Block statistics over all files
Every file is read once and cut into blocks; per block: mean, std, peak-to-peak, kurtosis
(Fisher, i.e. 0 for a Gaussian) and the median RPM. RMS is not shown: the signal sits on a ~3.9 V offset,
so RMS is almost equal to the mean. The table `blk` and the RPM traces are kept for sections 5–6.
The time axis is absolute (file start + offset) only for files with a known sampling rate.

In [ ]:
BLOCK_N = 50_000  # EDA choice: samples per block (1 s at 50 kHz)
RPM_STEP = 128  # RPM only changes on multiples of 128 samples (section 5), so every 128th value keeps the full trace

blk_rows, n_rows, rpm_trace = [], {}, {}
for p in FILES:
    v_, r_ = read_log(p)
    fs_ = sampling_rate(p)
    n_rows[p.name] = len(v_)
    rpm_trace[p.name] = r_[::RPM_STEP]
    nb = len(v_) // BLOCK_N
    b = v_[: nb * BLOCK_N].reshape(nb, BLOCK_N).astype(np.float64)
    rb = r_[: nb * BLOCK_N].reshape(nb, BLOCK_N)
    t0 = start_time(p)
    blk_rows.append(pd.DataFrame({
        "file": p.name, "block": np.arange(nb),
        "time": t0 + pd.to_timedelta(np.arange(nb) * BLOCK_N / fs_, unit="s") if fs_ else pd.NaT,
        "rpm": np.nanmedian(rb, axis=1), "rpm_range": np.nanmax(rb, axis=1) - np.nanmin(rb, axis=1),
        "mean": b.mean(1), "std": b.std(1),
        "p2p": b.max(1) - b.min(1), "kurtosis": kurtosis(b, axis=1),
    }))
    print(f"{p.name}: {len(v_):,} rows, {nb} blocks")
    del v_, r_, b, rb
blk = pd.concat(blk_rows, ignore_index=True)
files["rows"] = pd.Series(n_rows)
files["duration_min"] = files.rows / files.fs_hz / 60
files[["fs_hz", "rows", "duration_min", "duration_min_header", "gap_to_prev_s"]]

In [ ]:
# Per file: operating states and level of the signal.
blk.groupby("file").agg(blocks=("block", "size"), rpm_min=("rpm", "min"), rpm_max=("rpm", "max"),
                        mean_v=("mean", "median"), std_v=("std", "median"), p2p=("p2p", "median"),
                        kurtosis=("kurtosis", "median")).round(4)

In [ ]:
# Statistics over time for the logs with a time axis (Oct 12 and Oct 13), one column per day.
timed = blk.dropna(subset=["time"])
days = sorted(timed.time.dt.date.unique())
stats = ["rpm", "mean", "std", "p2p", "kurtosis"]
fig, ax = plt.subplots(len(stats), len(days), figsize=(7 * len(days), 2.2 * len(stats)), sharex="col", squeeze=False)
for j, d in enumerate(days):
    sub = timed[timed.time.dt.date == d]
    for i, s in enumerate(stats):
        ax[i, j].plot(sub.time, sub[s], ".", ms=1.5)
        ax[i, j].set_ylabel(s)
    ax[0, j].set_title(str(d))
plt.tight_layout(); plt.show()

In [ ]:
# Statistics against RPM (all files; each point is one block), coloured by file.
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for name, sub in blk.groupby("file"):
    for a, s in zip(ax, ["mean", "std", "p2p", "kurtosis"]):
        a.plot(sub.rpm, sub[s], ".", ms=2, label=name[4:-4])
for a, s in zip(ax, ["mean", "std", "p2p", "kurtosis"]):
    a.set_xlabel("RPM"); a.set_title(s)
ax[-1].legend(fontsize=6, markerscale=4, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

## 5. RPM
How often RPM is updated, the RPM over time for every file, and which operating points occur.

In [ ]:
# Run lengths of constant RPM in the file from section 3. If they are multiples of one number,
# that is the RPM update period in samples.
change = np.flatnonzero(np.diff(rpm) != 0) + 1
runs = np.diff(np.r_[0, change, len(rpm)])
print("most common run lengths (samples: count):", pd.Series(runs).value_counts().head(10).to_dict())
print("greatest common divisor of run lengths:", np.gcd.reduce(runs[1:-1]) if len(runs) > 2 else "n/a")

In [ ]:
# RPM over time, logs with a known sampling rate: clock time, one panel per day; each colour is one 15-min log file.
timed_files = [p for p in FILES if sampling_rate(p)]
days = sorted({start_time(p).date() for p in timed_files})
fig, ax = plt.subplots(len(days), 1, figsize=(12, 3 * len(days)), squeeze=False)
for a, d in zip(ax[:, 0], days):
    for p in [p for p in timed_files if start_time(p).date() == d]:
        r = rpm_trace[p.name]
        t = start_time(p) + pd.to_timedelta(np.arange(len(r)) * RPM_STEP / sampling_rate(p), unit="s")
        a.plot(t, r, lw=0.8)
    a.set_title(f"{d} (colour = log file)"); a.set_ylabel("RPM")
plt.tight_layout(); plt.show()

In [ ]:
# RPM over time, logs without a sampling rate: x-axis in samples (time unknown, FS_NO_HEADER), one panel per file.
untimed = [p for p in FILES if not sampling_rate(p)]
fig, ax = plt.subplots((len(untimed) + 1) // 2, 2, figsize=(12, 2.2 * ((len(untimed) + 1) // 2)), squeeze=False)
for a, p in zip(ax.flat, untimed):
    r = rpm_trace[p.name]
    a.plot(np.arange(len(r)) * RPM_STEP, r, lw=0.8)
    a.set_title(p.name, fontsize=9); a.set_ylabel("RPM")
for a in ax.flat[len(untimed):]:
    a.axis("off")
ax[-1, 0].set_xlabel("sample"); ax[-1, 1].set_xlabel("sample")
plt.tight_layout(); plt.show()

In [ ]:
# Operating points: number of 1-s blocks per (rounded) RPM, by day.
RPM_BIN = 10  # EDA choice: RPM bin width for grouping
blk["rpm_bin"] = (blk.rpm / RPM_BIN).round() * RPM_BIN
pd.crosstab(blk.rpm_bin, blk.file.str[4:14]).loc[lambda t: t.sum(axis=1) > 0]

## 6. Spectra and envelope spectra per operating point
For files with a known sampling rate, blocks are grouped by RPM bin (section 5). RPM jitters by about ±1
within a block, so a block is used when its RPM range is at most `RPM_TOL`.

- **Spectrum:** Welch spectrum of the raw signal, averaged over blocks.
- **Envelope spectrum:** each block is band-pass filtered, its envelope is taken (magnitude of the Hilbert
  transform), and the spectrum of the envelope is computed. Repetitive impacts (e.g. from a bearing defect or a
  rotating part) show up as peaks at their repetition frequency and its harmonics, even when the raw spectrum
  only shows a broad resonance. The envelope spectra are plotted against **order** = frequency / shaft frequency,
  with the shaft frequency from the logged RPM (RPM / 60) of each block. Peaks at integer orders are
  shaft-synchronous; peaks at a fixed non-integer order at every speed point to e.g. bearing elements.

The shaft-harmonic lines are reference lines at k × RPM / 60 (k = 1, 2, ...), computed from the `RPM` column;
they are not detected peaks. Bearing defect orders need the bearing type and geometry (**PLACEHOLDER**, section 7).

In [ ]:
WELCH = dict(window="hann", nperseg=2 ** 15, noverlap=2 ** 14, detrend="constant", scaling="density", average="mean")  # EDA choice
SPEC_FILES = [p for p in FILES if sampling_rate(p)]  # EDA choice: all files with a known fs
MAX_BLOCKS = 60  # EDA choice: at most this many blocks per (file, RPM bin), spread over the file
RPM_TOL = 5  # EDA choice: largest RPM range (max - min) inside a block that is still used

# Envelope settings. Bands are EDA choices: one broad high-pass band and the three resonance-like humps
# seen in the raw spectrum (~5.8, ~12, ~18 kHz). Change them after looking at the spectrum below.
ENV_BANDS = {"1-24 kHz": (1_000, 24_000), "5-7 kHz": (5_000, 7_000),
             "11-13 kHz": (11_000, 13_000), "17-19.5 kHz": (17_000, 19_500)}  # EDA choice
ENV_ORDER = 4  # EDA choice: Butterworth order, applied forward-backward (zero phase)
ENV_WELCH = dict(window="hann", nperseg=BLOCK_N, detrend="constant", scaling="density")  # EDA choice: one 1-s segment per block (1 Hz resolution)
ORDERS = np.arange(0, 20, 0.02)  # EDA choice: order axis the envelope spectra are put on (up to order 20)

spec, env_spec = {}, {}  # rpm_bin -> list of PSDs; (band, rpm_bin) -> list of envelope PSDs on ORDERS
for p in SPEC_FILES:
    v_, r_ = read_log(p)
    fs_ = sampling_rate(p)
    sos = {k: butter(ENV_ORDER, b, btype="bandpass", fs=fs_, output="sos") for k, b in ENV_BANDS.items()}
    sub = blk[(blk.file == p.name) & (blk.rpm_range <= RPM_TOL)]
    for rb, g in sub.groupby("rpm_bin"):
        pick = g.iloc[np.linspace(0, len(g) - 1, min(MAX_BLOCKS, len(g))).astype(int)]
        for b, rpm_b in zip(pick.block, pick.rpm):
            seg = v_[b * BLOCK_N:(b + 1) * BLOCK_N].astype(np.float64)
            f, P = welch(seg, fs=fs_, **WELCH)
            spec.setdefault(rb, []).append(P)
            for k in ENV_BANDS:
                env = np.abs(hilbert(sosfiltfilt(sos[k], seg - seg.mean())))
                fe, Pe = welch(env, fs=fs_, **ENV_WELCH)
                env_spec.setdefault((k, rb), []).append(np.interp(ORDERS, fe / (rpm_b / 60), Pe))
    del v_, r_
print("blocks per RPM bin:", {k: len(x) for k, x in spec.items()})

In [ ]:
# Raw spectrum, one row per RPM bin: full range and low-frequency zoom with shaft harmonics.
# Same y-axis in every row, so the rows can be compared.
rbs = sorted(spec)
fig, ax = plt.subplots(len(rbs), 2, figsize=(14, 2.2 * len(rbs)), sharex="col", sharey=True, squeeze=False)
for i, rb in enumerate(rbs):
    P = np.mean(spec[rb], axis=0)
    ax[i, 0].semilogy(f / 1e3, P, lw=0.6)
    ax[i, 1].semilogy(f, P, lw=0.8)
    for k in range(1, 11):
        ax[i, 1].axvline(k * rb / 60, color="grey", lw=0.5, ls=":")
    ax[i, 0].set_ylabel("PSD [V²/Hz]")
    ax[i, 0].set_title(f"RPM {rb:.0f} ({len(spec[rb])} blocks)", loc="left", fontsize=10)
ax[0, 1].set_xlim(0, 200)  # EDA choice: low-frequency zoom
ax[-1, 0].set_xlabel("frequency [kHz]")
ax[-1, 1].set_xlabel("frequency [Hz] (dotted: shaft harmonics 1-10 × RPM/60)")
plt.tight_layout(); plt.show()

In [ ]:
# Envelope spectra against order: one row per RPM bin, one column per band (same y-axis within a column).
# Dotted lines at integer orders (shaft harmonics).
m = ORDERS > 0.3  # skip the 0-Hz region of the envelope
fig, ax = plt.subplots(len(rbs), len(ENV_BANDS), figsize=(4.5 * len(ENV_BANDS), 2.2 * len(rbs)),
                       sharex=True, sharey="col", squeeze=False)
for i, rb in enumerate(rbs):
    for j, k in enumerate(ENV_BANDS):
        Pe = np.mean(env_spec[(k, rb)], axis=0)
        ax[i, j].semilogy(ORDERS[m], Pe[m], lw=0.7)
        for o in range(1, int(ORDERS[-1]) + 1):
            ax[i, j].axvline(o, color="grey", lw=0.4, ls=":")
        ax[0, j].set_title(f"band {k}", fontsize=10)
    ax[i, 0].set_ylabel(f"RPM {rb:.0f}\n({len(spec[rb])} blocks)")
for j in range(len(ENV_BANDS)):
    ax[-1, j].set_xlabel("order (dotted: integer orders)")
plt.tight_layout(); plt.show()

## 7. Open questions (to clarify with Jim-Tec / Arla)
- What sensor is `Volt_CH0` (accelerometer, AE, ultrasound), where is it mounted, and what are its sensitivity and conditioning?
- Sampling rate of the logs without a header (2026-04-14 to 2026-10-09)? (`FS_NO_HEADER`)
- Why does the signal level differ between days (mean ≈0 V with std ≈3 mV on 2026-09-23 and 2026-10-05, ≈2.4 V on 2026-04-14,
  ≈3.9 V on 2026-10-09/12/13)? Sensor bias, different setup, or sensor not connected on the ≈0 V days?
- How is RPM measured and how often is it updated? Is it the motor or the shaft at the bearing (gear ratio)?
- Bearing type and geometry (for defect orders), lubricant, and any maintenance or lubrication events in the logged period.
- The last 15-min log of 2026-10-13 is shorter than the others (~11.4 min): was logging stopped?